# Step 13 – Comparison with Hansen Global Forest Change: blind verification of disagreement pixels

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 2.7, 3.7 · Table 12
- **Needs:** CatBoost map tiles (`DATA_DIR/maps/CatBoost/`) and `DATA_DIR/P2_hansen_10m.tif`
- **Writes:** `RESULTS_DIR/R1_5b_verification/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / P2_hansen_10m.tif
#   DATA_DIR / maps/CatBoost/
#   RESULTS_DIR / R1_5b_verification/

In [ ]:
!pip install -q rasterio

In [ ]:
"""
Verification of CatBoost-only and Hansen-only forest pixels with VHR imagery

Run in a Colab notebook AFTER the CatBoost map is finished (DATA_DIR/maps/CatBoost/) and the
Hansen layers are exported on the same 10 m grid (DATA_DIR/P2_hansen_10m.tif).
In Colab first:  !pip install -q rasterio

The script works in two stages, decided automatically:

STAGE 1 (no sample yet) - draws the sample
  - CatBoost-only = forest in the CatBoost map, non-forest in Hansen
  - Hansen-only   = forest in Hansen, non-forest in the CatBoost map
  - simple random sample of N_CB_ONLY and N_HANSEN_ONLY pixels (seeded, reproducible)
  - the two samples are MIXED in random order and numbered P001, P002, ... so labelling is
    BLIND to the stratum; the stratum is kept in a separate key file
  Outputs (OUT_DIR):
    R1_5b_points.kml             open in Google Earth Pro: point + 10 m pixel square
    R1_5b_labelling_sheet.csv    fill in: label (1-6), imagery_date, confidence (H/L), notes
    R1_5b_KEY_do_not_open.csv    stratum of each point (do not open before labelling ends)
    R1_5b_strata_area.csv        area of each stratum and of forest agreement (km2)

STAGE 2 (labelling sheet filled in and uploaded back to OUT_DIR) - summarises
  - % of each land-cover category per stratum, with Wilson 95% confidence intervals
  Outputs: R1_5b_results.csv, R1_5b_results_grouped.csv

Label codes (use the same interpretation rules as for the reference data):
  1 = closed forest   2 = open woodland / matorral   3 = shrubland
  4 = olive grove / orchard   5 = crop / grass   6 = other (bare, urban, water)
"""


import os, glob
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer

# ------------------------------------------------------------------ config
MAP_DIR      = DATA_DIR + 'maps/CatBoost/'
HANSEN_PATH  = DATA_DIR + 'P2_hansen_10m.tif'
OUT_DIR      = RESULTS_DIR + 'R1_5b_verification/'
FOREST_CODE  = 1            # CatBoost class code for forest
NODATA       = 255
# >>> Hansen forest definition: set EXACTLY as in the original comparison (Table 5) <<<
HANSEN_TREECOVER_MIN = 30   # % tree cover in 2000
LOSS_UNTIL   = 24           # remove pixels lost 2001..(2000+LOSS_UNTIL); 0 = ignore loss
ADD_GAIN     = False        # True = count Hansen gain (2000-2012) pixels as forest
N_CB_ONLY     = 200
N_HANSEN_ONLY = 100
SEED          = 42
CATEGORIES = {1: 'Closed forest', 2: 'Open woodland / matorral', 3: 'Shrubland',
              4: 'Olive grove / orchard', 5: 'Crop / grass', 6: 'Other (bare, urban, water)'}
GROUPS = {1: 'Closed forest', 2: 'Other woody vegetation', 3: 'Other woody vegetation',
          4: 'Other woody vegetation', 5: 'Non-woody', 6: 'Non-woody'}

os.makedirs(OUT_DIR, exist_ok=True)
KEY_PATH, SHEET_PATH = OUT_DIR + 'R1_5b_KEY_do_not_open.csv', OUT_DIR + 'R1_5b_labelling_sheet.csv'
tiles = sorted(glob.glob(MAP_DIR + '*.tif'))
assert tiles, f'No CatBoost map tiles in {MAP_DIR}'
hansen = rasterio.open(HANSEN_PATH)
assert abs(hansen.transform.a - 10) < 1e-6, 'Hansen raster must be on the 10 m grid'

def strata(tile_path):
    """CatBoost-only and Hansen-only masks (and agreement counts) for one map tile."""
    with rasterio.open(tile_path) as src:
        cb, tr = src.read(1), src.transform
    assert src.crs == hansen.crs and abs(tr.a - hansen.transform.a) < 1e-6, 'Grids differ'
    win = from_bounds(*src.bounds, transform=hansen.transform)
    off = np.array([win.col_off, win.row_off])
    assert np.allclose(off, np.round(off), atol=1e-3), 'CatBoost and Hansen pixels are not aligned'
    win = win.round_offsets().round_lengths()
    h = hansen.read(window=win, boundless=True, fill_value=NODATA)[:, :cb.shape[0], :cb.shape[1]]
    tc, loss, gain, dmask = h
    valid = (cb != NODATA) & (tc != NODATA) & (dmask == 1)          # mapped land pixels
    h_forest = tc >= HANSEN_TREECOVER_MIN
    if LOSS_UNTIL:
        h_forest &= ~((loss >= 1) & (loss <= LOSS_UNTIL))
    if ADD_GAIN:
        h_forest |= gain == 1
    c_forest = cb == FOREST_CODE
    masks = {'CatBoost-only': valid & c_forest & ~h_forest, 'Hansen-only': valid & h_forest & ~c_forest}
    agree = {'Both forest': valid & c_forest & h_forest, 'Both non-forest': valid & ~c_forest & ~h_forest}
    return masks, agree, tr, cb, tc, loss

# ================================================================== STAGE 2: summarise labels
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return 100 * (c - h), 100 * (c + h)

sheet_done = False
if os.path.exists(SHEET_PATH):
    # sep=None: accepts comma- or semicolon-separated files (French Excel saves with ';')
    sheet = pd.read_csv(SHEET_PATH, encoding='utf-8-sig', sep=None, engine='python')
    sheet.columns = sheet.columns.str.strip()
    sheet_done = len(sheet) > 0 and pd.to_numeric(sheet['label'], errors='coerce').notna().all()

if sheet_done:
    key = pd.read_csv(KEY_PATH)
    lab = sheet[['point_id', 'label', 'confidence']].merge(key[['point_id', 'stratum']], on='point_id')
    lab['label'] = lab['label'].astype(int)
    assert lab['label'].isin(CATEGORIES).all(), 'Labels must be codes 1-6'
    rows, grows = [], []
    for s, g in lab.groupby('stratum'):
        n = len(g)
        for code, name in CATEGORIES.items():
            k = int((g['label'] == code).sum())
            lo, hi = wilson(k, n)
            rows.append({'Stratum': s, 'Category': name, 'n': k, '%': round(100 * k / n, 1),
                         '95% CI': f'{lo:.1f}-{hi:.1f}'})
        for grp in dict.fromkeys(GROUPS.values()):
            k = int(g['label'].map(GROUPS).eq(grp).sum())
            lo, hi = wilson(k, n)
            grows.append({'Stratum': s, 'Group': grp, 'n': k, '%': round(100 * k / n, 1),
                          '95% CI': f'{lo:.1f}-{hi:.1f}'})
        print(f"{s}: {n} points, {100 * (g['confidence'].astype(str).str.upper() == 'L').mean():.0f}% "
              f"labelled with low confidence")
    res, gres = pd.DataFrame(rows), pd.DataFrame(grows)
    res.to_csv(OUT_DIR + 'R1_5b_results.csv', index=False)
    gres.to_csv(OUT_DIR + 'R1_5b_results_grouped.csv', index=False)
    print('\n=== Land cover of disagreement pixels (VHR interpretation) ===\n', res.to_string(index=False))
    print('\n=== Grouped ===\n', gres.to_string(index=False))
    print('\nFiles saved to:', OUT_DIR)

# ================================================================== STAGE 1: draw the sample
elif not os.path.exists(KEY_PATH):
    print(f'Hansen forest = tree cover 2000 >= {HANSEN_TREECOVER_MIN}%'
          + (f', minus loss 2001-{2000 + LOSS_UNTIL}' if LOSS_UNTIL else '')
          + (', plus gain' if ADD_GAIN else ''))
    # pass 1: count stratum pixels per tile
    counts, area = {}, {}
    for t in tiles:
        masks, agree, *_ = strata(t)
        counts[t] = {s: int(m.sum()) for s, m in masks.items()}
        for s, m in {**masks, **agree}.items():
            area[s] = area.get(s, 0) + int(m.sum())
    area_km2 = pd.Series({s: n * 100 / 1e6 for s, n in area.items()}, name='km2').round(2)
    area_km2.to_csv(OUT_DIR + 'R1_5b_strata_area.csv')
    print('\nArea by agreement class (km2):\n', area_km2.to_string())

    # pass 2: simple random sample of pixel indices across all tiles, per stratum
    rng = np.random.default_rng(SEED)
    wanted = {'CatBoost-only': N_CB_ONLY, 'Hansen-only': N_HANSEN_ONLY}
    picks = {}
    for s, n in wanted.items():
        total = sum(c[s] for c in counts.values())
        assert total >= n, f'Only {total} {s} pixels'
        picks[s] = np.sort(rng.choice(total, size=n, replace=False))
    samples, start = [], {s: 0 for s in wanted}
    for t in tiles:
        masks, _, tr, cb, tc, loss = strata(t)
        for s in wanted:
            n_t = counts[t][s]
            sel = picks[s][(picks[s] >= start[s]) & (picks[s] < start[s] + n_t)] - start[s]
            start[s] += n_t
            if sel.size:
                r, c = np.nonzero(masks[s])
                r, c = r[sel], c[sel]
                x, y = tr * (c + 0.5, r + 0.5)                      # pixel centres
                samples.append(pd.DataFrame({'stratum': s, 'x': x, 'y': y, 'catboost_class': cb[r, c],
                                             'hansen_treecover2000': tc[r, c], 'hansen_lossyear': loss[r, c],
                                             'tile': os.path.basename(t)}))
    smp = pd.concat(samples, ignore_index=True)
    smp = smp.sample(frac=1, random_state=SEED).reset_index(drop=True)   # mix strata -> blind
    smp.insert(0, 'point_id', [f'P{i + 1:03d}' for i in range(len(smp))])
    to_ll = Transformer.from_crs(hansen.crs, 'EPSG:4326', always_xy=True)
    smp['lon'], smp['lat'] = to_ll.transform(smp['x'].values, smp['y'].values)
    smp.to_csv(KEY_PATH, index=False)

    sheet = smp[['point_id', 'lon', 'lat']].copy()
    for col in ['label', 'imagery_date', 'confidence', 'notes']:
        sheet[col] = ''
    sheet.to_csv(SHEET_PATH, index=False, encoding='utf-8-sig')         # Excel-friendly

    legend = '<br>'.join(f'{k} = {v}' for k, v in CATEGORIES.items())
    marks = []
    for _, p in smp.iterrows():
        sq = [(p.x - 5, p.y - 5), (p.x + 5, p.y - 5), (p.x + 5, p.y + 5), (p.x - 5, p.y + 5), (p.x - 5, p.y - 5)]
        ring = ' '.join(f'{lo:.7f},{la:.7f},0' for lo, la in (to_ll.transform(a, b) for a, b in sq))
        marks.append(f"""<Placemark><name>{p.point_id}</name><styleUrl>#px</styleUrl>
<description><![CDATA[Label the 10 m square:<br>{legend}]]></description>
<MultiGeometry><Point><coordinates>{p.lon:.7f},{p.lat:.7f},0</coordinates></Point>
<Polygon><outerBoundaryIs><LinearRing><coordinates>{ring}</coordinates></LinearRing></outerBoundaryIs></Polygon>
</MultiGeometry></Placemark>""")
    kml = f"""<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2"><Document><name>Hansen verification points</name>
<Style id="px"><IconStyle><scale>0.6</scale></IconStyle><LabelStyle><scale>0.8</scale></LabelStyle>
<LineStyle><color>ff00ffff</color><width>2</width></LineStyle><PolyStyle><fill>0</fill></PolyStyle></Style>
{''.join(marks)}
</Document></kml>"""
    with open(OUT_DIR + 'R1_5b_points.kml', 'w', encoding='utf-8') as f:
        f.write(kml)
    print(f'\nSample: {len(smp)} points ({N_CB_ONLY} CatBoost-only + {N_HANSEN_ONLY} Hansen-only), mixed and numbered.')
    print('Download R1_5b_points.kml and R1_5b_labelling_sheet.csv. Do NOT open the KEY file before labelling.')
    print('Files saved to:', OUT_DIR)

else:
    print('A sample already exists. Fill in every "label" in R1_5b_labelling_sheet.csv, upload it back to',
          OUT_DIR, 'with the same name, and run this cell again to get the results.')

In [ ]:
import pandas as pd
D = RESULTS_DIR + 'R1_5b_verification/'
key = pd.read_csv(D + 'R1_5b_KEY_do_not_open.csv')
sh = pd.read_csv(D + 'R1_5b_labelling_sheet.csv', encoding='utf-8-sig', sep=None, engine='python')
d = sh.merge(key[['point_id', 'stratum']], on='point_id')
d = d[d['confidence'].astype(str).str.strip().str.upper() == 'H']
groups = {1: 'Closed forest', 2: 'Other woody vegetation', 3: 'Other woody vegetation',
          4: 'Other woody vegetation', 5: 'Non-woody', 6: 'Non-woody'}
for s, g in d.groupby('stratum'):
    print(f'\n{s}: {len(g)} high-confidence points')
    print((g['label'].astype(int).map(groups).value_counts(normalize=True) * 100).round(1).to_string())